## INGESTION PIPELINE :

In [30]:
# Data => Documents
import os
from langchain_community.document_loaders import PyPDFLoader

In [31]:
def load_allpdfs():
    folder_path = "data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path,filename)
    
            loader = PyPDFLoader(pdf_path)
            docs = loader.load()
    
            all_docs.extend(docs)
            num_docs += 1

    print("total pdfs : ", num_docs)
    print("total pages : ", len(all_docs))

    return all_docs

In [32]:
all_pdf_document = load_allpdfs()

total pdfs :  2
total pages :  32


### chunks

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def splits_docs(document, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(document)
    return chunked_docs

In [34]:
chunks = splits_docs(all_pdf_document)

In [35]:
len(chunks)

321

### Embedding

In [36]:
from sentence_transformers import SentenceTransformer

In [37]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading model.....", model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimension", self.model.get_sentence_embedding_dimension())

    def generate_embedding(self,text):
        embedding = self.model.encode(text,show_progress_bar=True)
        print("embedding shape", embedding.shape)
        return embedding

In [38]:
embedding_manager = EmbeddingManager()

Loading model..... all-MiniLM-L6-v2


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2469.21it/s]


embedding dimension 384


C:\Users\ASUS\AppData\Local\Temp\ipykernel_8408\40710886.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimension", self.model.get_sentence_embedding_dimension())


### Vector Store

In [39]:
import chromadb
import uuid

In [43]:
class VectorStoreManager:
    def __init__(self, persist_directory="Data/vector_store", collection_name="pdf_documnets"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create a collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description" : "vectore store collection for pdf embedding in RAG"}
        )

        print("initialized the vectore store with collection", self.collection_name)
        print("docs in collection", self.collection.count())

    # function for store embedding and documents in vectore store

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("length of documents and embedding is not match")

        # store => idx, doucments, embedding, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (docs, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(docs.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(docs.page_content)

            documents_content.append(docs.page_content)
            all_metadata.append(metadata)

            embeddings_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )
            
        

In [44]:
vector_store = VectorStoreManager()

initialized the vectore store with collection pdf_documnets
docs in collection 642


In [45]:
texts = [docs.page_content for docs in chunks]

embedding = embedding_manager.generate_embedding(texts)
vector_store.add_documents(chunks, embedding)

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 11/11 [00:22<00:00,  2.04s/it]


embedding shape (321, 384)


# Retrieval Pipeline

In [46]:
from sklearn.metrics.pairwise import cosine_similarity

In [47]:
class RAGRetriever:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        # semantic search
        result = self.vector_store.collection.query(
            query_embeddings = [query_embedding.tolist()],
            n_results = top_k
        )

        # cosine similarity
        retrieved_docs = []
        if result["documents"] and result["documents"][0]:
            ids = result["ids"][0]
            metadatas = result["metadatas"][0]
            documents = result["documents"][0]
            distances = result["distances"][0]

            for i , (docs_id, metadata, document, distance) in enumerate(zip(ids,metadatas,documents,distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "ids" : docs_id,
                        "metadata" : metadata,
                        "document" : document,
                        "similarity_score" : similarity_score,
                        "distance" : distance,
                        "rank" : i+1
                    })
                    
            print(f"retrieved {len(retrieved_docs)} documents")
            
        else:
            print("no documents found")

        return retrieved_docs

In [48]:
rag_retriever = RAGRetriever(embedding_manager,vector_store)

In [49]:
rag_retriever.retrieve("what is encoder decoder?")

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 19.90it/s]

embedding shape (1, 384)
retrieved 5 documents


[{'ids': 'doc_581a4a6c-52bf-4dff-9e94-873708c0a27c',
  'metadata': {'publisher': 'Curran Associates, Inc.',
   'producer': 'pdfcpu v0.9.1 dev',
   'source': 'data\\research1.pdf',
   'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'language': 'en-US',
   'doc_index': 18,
   'type': 'Conference Proceedings',
   'firstpage': '5998',
   'content_length': 447,
   'created': '2017',
   'eventtype': 'Poster',
   'date': '2017',
   'book': 'Advances in Neural Information Processing Systems 30',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the enc

# Integrate with LLMs

## Using Groq 

In [ ]:
import os

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)

In [52]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we not found relevant context for this query")

    # context + query
    prompt = f"""  use given context to generate the answer for the query
                   Context: {context}
                   Query: {query} """
    response = llm.invoke([prompt.format(context=context, query=query)])   # expecting a list as prompt
    return response.content

In [53]:
answer = generate_output("what is encoder-decoder ?", rag_retriever, llm)

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 26.26it/s]


embedding shape (1, 384)
retrieved 3 documents


In [54]:
print(answer)

<think>
Okay, the user is asking, "what is encoder-decoder?" Let me start by recalling the context provided. The context talks about the encoder and decoder in a transformer model. The encoder has layers with sub-layers, and the decoder has similar layers but with an additional sub-layer for multi-head attention on the encoder's output.

First, I need to define what an encoder-decoder architecture is in general. It's a framework where an encoder processes input data into a context vector, and the decoder generates output based on that. In the context given, both encoder and decoder have 6 layers each. The encoder's layers include self-attention and feed-forward sub-layers, while the decoder adds a third sub-layer that attends to the encoder's output. Both use residual connections and layer normalization.

Wait, the user might not know about transformers specifically. Should I mention that this is in the context of transformer models? Probably, since the context is about transformers. S